# 3차 실험 Colab 스모크 테스트: Faster R-CNN (torchvision)
2차 방법(v1~v4) 하나를 골라 클래스 불균형 대응 조건 세 가지를 학습하고 val로 채점합니다. 3차에서 바뀌는 곳이 어디인지 보여 주고, Colab에서 그 부분이 동작하는지 확인하는 노트북입니다.

| 조건 | train 목록 | 손실 가중치 | epoch |
| --- | --- | --- | --- |
| `sampling` | `SAMPLING` 목록 | 없음 | round(E × N ÷ 목록 줄 수) |
| `weight` | 원래 train | `CLS_PW` | E |
| `both` | `SAMPLING` 목록 | `CLS_PW` | round(E × N ÷ 목록 줄 수) |

- E는 비교할 2차 기준 run의 epoch 수(`BASE_EPOCHS`), N은 원래 train 장수입니다. 총 학습량(step 수)을 기준 run과 맞춰서, 샘플링 효과와 학습량 효과가 섞이지 않게 합니다.
- 아무것도 하지 않는 기준 조건은 다시 학습하지 않고, 같은 방법·모델의 2차 run을 씁니다.
- 방법, 목록, 가중치 식의 자세한 설명은 [balance.py](../../../src/pill_detection/balance.py) 모듈 docstring에 있습니다.
- 손실 가중치는 `use_fastrcnn_class_weights()`가 학습에 쓰는 목록의 객체 수로 계산해 RoI 분류 손실에 겁니다. 클래스 가중치는 Ultralytics `cls_pw`와 같은 식이고, 배경(0번) 가중치는 알약 RoI 전체와 배경의 비중이 가중치가 없을 때와 같도록 정합니다.
- `both`의 가중치는 샘플링 후 분포로 셉니다. Ultralytics `cls_pw`와 같은 기준이고, 원래 분포로 세면 같은 불균형을 두 번 보정합니다.

**준비**
- 내 Drive의 `.env`에 `DRIVE_BASE`(pill data 폴더 경로)와 `WANDB_API_KEY`를 적습니다. 키 이름은 [.env.example](../../../.env.example)에서 확인합니다.
- `DRIVE_BASE/data/`에 zip이 있어야 합니다: `kaggle.zip`, `aihub_annotations.zip`, `aihub_images.zip`, 제출할 때만 `test_images.zip`
- 데이터가 팀과 같은지 확인(`make_dataset_lists.py --check`)은 2차 스모크 테스트에서 마쳤다고 봅니다.

**쓰는 법**
1. **각자 바꿀 값** 셀을 채우고 위에서부터 실행합니다. 학습 셀이 `CONDITIONS`의 조건을 차례로 학습하고 val로 채점합니다.
2. 먼저 `SMOKE = True`로 돌려 W&B 기록, Drive 저장, 조건 적용 확인 출력을 봅니다. run 이름이 `smoke_`로 시작하고 태그 `smoke`가 붙으니, 확인이 끝나면 W&B 실행과 Drive 폴더를 지워도 됩니다.
3. `SMOKE = False`로 바꾸고 그 셀부터 다시 실행해 본 학습을 돌립니다. 결과는 `DRIVE_BASE/<SAVE_SUBDIR>/<방법>_<모델>_<조건>`에 남습니다. (예: `v3_frcnn_rfs`, `v3_frcnn_cls0.5`, `v3_frcnn_rfs_cls0.5`)
4. 2차 run과 비교는 [exp3_model_selection.ipynb](exp3_model_selection.ipynb)의 `RUNS`에 이 run 폴더들을 넣어서 봅니다.

> **본 학습은 2차 기준 run을 만든 노트북의 학습 설정으로 돌립니다.** 이 노트북의 학습 루프는 2차 스모크 테스트와 같은 간단한 SGD 루프라서, 2차 기준 run(02 노트북: AdamW, 실효 batch 16, AMP 등)과 결과를 비교할 수 없습니다.
>
> 3차에서 바뀌는 곳은 `# [3차]` 표시가 있는 곳뿐입니다. 02 노트북을 복사해 3차 노트북을 만들 때 이 부분을 옮기고, 세 가지를 더 고칩니다.
> - `code_identity()`가 해시하는 공용 파일 목록에 `balance.py`를 넣습니다.
> - epoch 검사(5, 10, 20만 허용)가 조정한 epoch도 받게 합니다.
> - config의 `warmup_epochs`(3)를 epoch 수와 같은 비율로 3 × N ÷ 목록 줄 수로 바꿉니다. warmup은 epoch 단위라 1 epoch 길이가 바뀌면 warmup step 수도 바뀝니다. cosine 구간은 전체 step 기준이라 그대로 둡니다. (이 노트북의 SGD 루프에는 warmup이 없습니다)


In [ ]:
# 각자 바꿀 값. 값을 바꾸면 이 셀부터 다시 실행한다
ENV_FILE = '/content/drive/MyDrive/<내 .env 경로>'   # 내 Drive의 .env. DRIVE_BASE, WANDB_API_KEY가 들어 있어야 한다
SAVE_SUBDIR = '<cj_runs, jw_runs 또는 rw_runs>'   # 결과를 저장할 폴더(DRIVE_BASE 아래)
BRANCH = 'feat/41-exp3-balance'                  # balance.py가 있는 브랜치. main에 병합되면 'main'으로 바꾼다

VARIANT = 'v3'          # 기반이 되는 2차 방법: v1, v2, v3, v4 중 하나
BASE_EPOCHS = 20        # 비교할 2차 기준 run의 epoch 수(E)
MODEL = 'frcnn'         # run 이름 <방법>_<모델>_<조건>의 <모델> 부분에 에 들어간다
TRAIN = dict(lr0=0.005, momentum=0.9, weight_decay=1e-4, batch=4, seed=0)   # 스모크용 SGD 설정

SAMPLING = 'rfs'        # 'rfs'(오버샘플링) 또는 'under150'처럼 'under<클래스별 목표 객체 수>'(언더샘플링)
CLS_PW = 0.5            # 클래스 빈도 손실 가중치 세기(0~1). 0이면 가중치 없음, 1이면 객체 수에 반비례
CONDITIONS = ['sampling', 'weight', 'both']   # 돌릴 조건들. 3개를 모두 넣으면 세 조건을 한 번에 순차적으로 학습돌린다. 하나의 조건만 하고싶으면 ['weight'] 처럼 줄인다
SMOKE = True            # True: 조건마다 1 epoch, train 5%로 기록·저장·조건 적용만 확인한다. 본 학습은 False

In [ ]:
# 세션 시작 (Drive 연결, .env 읽기, 패키지 설치, 레포 받기, W&B 로그인)
import os, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pycocotools==2.0.11 wandb==0.31.0 python-dotenv==1.2.4
from dotenv import load_dotenv

if not Path(ENV_FILE).is_file():
    raise FileNotFoundError(f'.env가 없습니다: {ENV_FILE}\n'
                            '[해결] 내 Drive에 .env를 올리고(.env.example 참고), 각자 바꿀 값 셀의 ENV_FILE을 그 경로로 고친 뒤 그 셀부터 다시 실행하세요.')
load_dotenv(ENV_FILE, override=True)
DRIVE_BASE = Path(os.environ.get('DRIVE_BASE', '').strip())
if not os.environ.get('DRIVE_BASE', '').strip() or not DRIVE_BASE.is_dir():
    raise FileNotFoundError(f'DRIVE_BASE가 비어 있거나 Drive에 없는 폴더입니다: "{os.environ.get("DRIVE_BASE", "")}"\n'
                            '[해결] .env에 DRIVE_BASE=<pill data 폴더 경로>를 적으세요. 예) DRIVE_BASE=/content/drive/MyDrive/pill data')
if '<' in SAVE_SUBDIR:
    raise ValueError('SAVE_SUBDIR를 정하지 않았습니다.\n[해결] 각자 바꿀 값 셀의 SAVE_SUBDIR를 내 결과 폴더 이름으로 고치세요.')
SAVE_DIR = DRIVE_BASE / SAVE_SUBDIR

REPO = '/content/pill-object-detection'
!test -d {REPO} || git clone -q https://github.com/codeit-ai-15th/pill-object-detection.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git switch -q -C {BRANCH} FETCH_HEAD && git log -1 --format='%h %s'
sys.path.insert(0, f'{REPO}/src')

import wandb
os.environ['WANDB_ENTITY'] = 'codeit-prj1-team2'   # W&B 팀. .env의 값보다 이 값을 쓴다
wandb.login()   # .env의 WANDB_API_KEY로 로그인한다

In [ ]:
# 데이터 풀기와 2차 학습 폴더 만들기 (세션마다). 이번 세션에서 이미 만들었으면 건너뛴다
DS = f'/content/exp2/{VARIANT}'
if not Path(DS).is_dir():
    for z in ['kaggle', 'aihub_annotations', 'aihub_images']:
        !unzip -q -o "{DRIVE_BASE}/data/{z}.zip" -d {REPO}
    !PYTHONPATH=src python scripts/build_dataset.py --variant {VARIANT} --out {DS} --format coco

In [ ]:
# 조건별 train 목록과 epoch. 목록은 입력과 seed가 같으면 늘 같아서 세션마다 다시 만들어도 된다
import pandas as pd
from pill_detection.balance import load_train_objects, repeat_list, undersample_list, write_train_list

objects = load_train_objects(DS)
if SAMPLING == 'rfs':
    files = repeat_list(objects, seed=0)                    # 오버샘플링: 희소 클래스가 든 사진을 더 반복한다
elif SAMPLING.startswith('under') and SAMPLING[5:].isdigit():
    files = undersample_list(objects, int(SAMPLING[5:]))   # 언더샘플링: 클래스별 목표 객체 수까지만 사진을 남긴다
else:
    raise ValueError(f"SAMPLING이 맞지 않습니다: {SAMPLING}\n[해결] 'rfs', 'under150', 'under200' 중 하나를 적으세요.")
info = write_train_list(DS, SAMPLING, files)   # DS/sampling/<SAMPLING>/에 train.txt를 쓴다


def plan(tag, train_list, n_rows, cls_pw):
    """조건 하나의 학습 계획. epoch는 총 학습량(step 수)이 2차 기준 run과 같아지게 정한다."""
    return {'name': f'{VARIANT}_{MODEL}_{tag}', 'train_list': train_list, 'n_rows': n_rows, 'cls_pw': cls_pw,
            'epochs': round(BASE_EPOCHS * info['n_base'] / n_rows)}


PLANS = {
    'sampling': plan(SAMPLING, f'{DS}/sampling/{SAMPLING}/train.txt', info['n_list'], 0.0),
    'weight': plan(f'cls{CLS_PW}', None, info['n_base'], CLS_PW),
    'both': plan(f'{SAMPLING}_cls{CLS_PW}', f'{DS}/sampling/{SAMPLING}/train.txt', info['n_list'], CLS_PW),
}
if set(CONDITIONS) - set(PLANS):
    raise ValueError(f'모르는 조건이 있습니다: {sorted(set(CONDITIONS) - set(PLANS))}\n[해결] CONDITIONS를 sampling, weight, both 중에서 고르세요.')

RUN_CONDITIONS = CONDITIONS   # Faster R-CNN은 세 조건 모두 돌릴 수 있다
display(pd.DataFrame(PLANS).T.loc[RUN_CONDITIONS, ['name', 'n_rows', 'epochs', 'cls_pw']])

In [ ]:
# 채점 기록 함수: 점수와 조건을 run 폴더(Drive)에 남기고, 학습 때의 W&B 실행 Summary에 exp3/ 점수를 붙인다
import json, shutil
from pill_detection.balance import imbalance_summary


def find_run_id(run_dir):
    """run 폴더의 wandb/ 기록에서 학습 때의 W&B 실행 id를 찾는다. 여러 개면 가장 최근 것."""
    found = max(Path(run_dir, 'wandb').glob('*run-*/run-*.wandb'), key=lambda f: f.stat().st_mtime, default=None)
    if found is None:
        raise FileNotFoundError(f'W&B 기록이 없습니다: {run_dir}/wandb/\n[해결] 학습할 때 W&B 실행의 dir을 run 폴더로 줬는지 확인하세요.')
    return found.stem[4:]


def save_scores(run_dir, scores, condition):
    """val 점수 3개와 희소/빈번 절반 AP(gap)를 출력·저장한다. 2차 run과의 비교는 exp3_model_selection.ipynb가 scores.json으로 한다."""
    summary = {k: scores[k] for k in ('val', 'kaggle', 'test_only')} | imbalance_summary(scores['per_class'], f'{DS}/classes.csv', VARIANT)
    print(Path(run_dir).name, '|', '  '.join(f'{k} {v:.4f}' for k, v in summary.items() if k != 'n_classes'))
    Path(run_dir, 'scores.json').write_text(json.dumps(scores, indent=2), encoding='utf-8')
    Path(run_dir, 'condition.json').write_text(json.dumps(condition, indent=2), encoding='utf-8')
    shutil.copy(f'{DS}/dataset_info.json', run_dir)   # /content는 세션이 끝나면 지워지므로 Drive에 남긴다
    with wandb.init(entity='codeit-prj1-team2', project='pill-object-detection', id=find_run_id(run_dir),
                    resume='allow', dir=str(run_dir)) as run:
        run.summary.update({f'exp3/{k}': v for k, v in summary.items()})
        run.config.update({'exp3': condition})

In [ ]:
# 모델과 데이터 (2차 스모크 테스트의 Torchvision 준비 셀과 같다. [3차] 표시만 새로 들어갔다)
import torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CocoDetection
from torchvision.models.detection import fasterrcnn_resnet50_fpn, roi_heads
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_convert
from torchvision.transforms.functional import to_tensor
from pill_detection.utils import xyxy_to_xywh

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cats = json.load(open(f'{DS}/coco/train.json', encoding='utf-8'))['categories']
label_of = {c['id']: c['idx'] + 1 for c in cats}     # K-코드 → 학습 label (0은 배경)
category_of = {v: k for k, v in label_of.items()}    # 예측 label → K-코드 (채점에 씀)


class PillDataset(CocoDetection):
    """COCO 파일을 Faster R-CNN 입력(이미지 tensor, xyxy boxes, labels)으로 읽는다."""

    def __getitem__(self, index):
        image, anns = super().__getitem__(index)
        boxes = torch.tensor([a['bbox'] for a in anns], dtype=torch.float32).reshape(-1, 4)   # COCO bbox는 xywh
        labels = torch.tensor([label_of[a['category_id']] for a in anns], dtype=torch.int64)
        return to_tensor(image), {'boxes': box_convert(boxes, 'xywh', 'xyxy'), 'labels': labels}

    def file_name(self, index):
        return self.coco.imgs[self.ids[index]]['file_name']


def train_dataset(train_list):
    """[3차] train Dataset. train_list(train.txt)를 주면 그 줄 순서대로 읽고, 반복된 줄은 반복해서 읽는다."""
    dataset = PillDataset(f'{DS}/images/train', f'{DS}/coco/train.json')
    if train_list:
        id_of = {image['file_name']: image_id for image_id, image in dataset.coco.imgs.items()}
        dataset.ids = [id_of[Path(line).name] for line in Path(train_list).read_text(encoding='utf-8').splitlines()]
    return dataset


def collate(batch):
    return tuple(zip(*batch))   # 이미지 크기가 달라도 묶이도록 tensor로 쌓지 않는다


def build_model():
    """COCO로 사전학습한 Faster R-CNN의 분류기를 모델 클래스 수(+배경)로 바꾼다."""
    model = fasterrcnn_resnet50_fpn(weights='DEFAULT', box_score_thresh=0.001)   # 채점용으로 점수 낮은 예측도 남긴다
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, len(cats) + 1)
    return model.to(DEVICE)


def fit(train_set, epochs, run_dir, tags):
    """SGD로 학습하고 W&B에 기록한다. 스모크면 train의 5%만 쓴다. 체크포인트는 epoch마다 weights/last.pt에 덮어쓴다."""
    torch.manual_seed(TRAIN['seed'])
    if SMOKE:
        train_set = Subset(train_set, torch.randperm(len(train_set))[:round(len(train_set) * 0.05)].tolist())
    loader = DataLoader(train_set, batch_size=TRAIN['batch'], shuffle=True, num_workers=2, collate_fn=collate)
    model = build_model()
    optimizer = torch.optim.SGD([q for q in model.parameters() if q.requires_grad], lr=TRAIN['lr0'],
                                momentum=TRAIN['momentum'], weight_decay=TRAIN['weight_decay'])
    (run_dir / 'weights').mkdir(parents=True)   # W&B는 dir이 없으면 임시 폴더에 기록하므로 먼저 만든다
    wandb.init(entity='codeit-prj1-team2', project='pill-object-detection', name=run_dir.name, dir=str(run_dir),
               tags=tags, config={**TRAIN, 'epochs': epochs})
    for epoch in range(1, epochs + 1):
        model.train()
        for images, targets in loader:
            losses = model([i.to(DEVICE) for i in images], [{k: v.to(DEVICE) for k, v in t.items()} for t in targets])
            loss = sum(losses.values())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            wandb.log({'epoch': epoch, 'train/loss': loss.item(), **{f'train/{k}': v.item() for k, v in losses.items()}})
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': epoch}, run_dir / 'weights' / 'last.pt')
    wandb.finish()
    return model


@torch.no_grad()
def predict(model, dataset, batch_size=8):
    """dataset의 이미지를 예측해 채점 형식 dict(file_name, category_id=K-코드, bbox=xywh, score)의 목록으로 돌려준다."""
    model.eval()
    predictions, index = [], 0
    for images, _ in DataLoader(dataset, batch_size=batch_size, num_workers=2, collate_fn=collate):
        for out in model([image.to(DEVICE) for image in images]):
            file_name = dataset.file_name(index)
            index += 1
            for box, label, score in zip(out['boxes'].tolist(), out['labels'].tolist(), out['scores'].tolist()):
                predictions.append({'file_name': file_name, 'category_id': category_of[label], 'bbox': xyxy_to_xywh(box), 'score': score})
    return predictions


val_set = PillDataset(f'{DS}/images/val', f'{DS}/coco/val.json')
print(f'모델 클래스 {len(cats)}개 + 배경, val {len(val_set):,}장, {DEVICE}')

## 학습과 채점
- `CONDITIONS`의 조건을 차례로 학습하고, 마지막 epoch의 모델로 val을 예측해 채점합니다.
- 다시 실행하면 채점까지 끝난 조건은 건너뜁니다. 이 노트북은 끊긴 run을 이어서 학습하지 않습니다.
- 학습 전에 조건 적용 확인을 출력합니다: train 장수가 목록 줄 수와 같은지, `cls_pw`를 준 조건에만 손실 가중치가 걸렸는지 보고, 건 가중치(배경, 클래스 범위)를 출력합니다. (v3 `weight` 조건: 배경 0.887, 클래스 0.48~3.30)


In [ ]:
# 조건별 학습과 val 채점
from datetime import datetime
from pill_detection.balance import use_fastrcnn_class_weights
from pill_detection.evaluate import evaluate_val


def check_applied(train_set, p):
    """조건 적용 확인: train 장수가 목록 줄 수와 같고, cls_pw를 준 조건에만 분류 손실이 가중치 손실로 바뀌어 있다. 건 가중치도 출력한다."""
    weight = getattr(roi_heads.fastrcnn_loss, 'weight', None)   # [배경, 클래스...]. use_fastrcnn_class_weights가 바꿔 끼운 손실 함수에만 있다
    shown = f'배경 {weight[0]:.3f}, 클래스 {weight[1:].min():.2f}~{weight[1:].max():.2f}' if weight is not None else '없음'
    print(f"{p['name']}: train {len(train_set):,}장 (예상 {p['n_rows']:,}), 손실 가중치 {shown}")
    if len(train_set) != p['n_rows'] or (weight is not None) != bool(p['cls_pw']):
        raise RuntimeError('조건이 학습에 그대로 들어가지 않았습니다.\n[해결] 학습 셀의 [3차] 줄과 조건 셀의 PLANS를 확인하세요.')


for condition in RUN_CONDITIONS:
    p = PLANS[condition]
    run_dir = SAVE_DIR / (f"smoke_{p['name']}_{datetime.now():%m%d_%H%M}" if SMOKE else p['name'])
    if (run_dir / 'scores.json').exists():
        print(f'{run_dir.name}: 채점까지 끝난 run이라 건너뜁니다.')
        continue
    if run_dir.exists():
        raise FileExistsError(f'{run_dir}는 채점 전에 멈춘 run입니다.\n[해결] 이 노트북은 이어서 학습하지 않습니다. Drive에서 이 폴더를 지우고 다시 실행하세요.')
    train_set = train_dataset(p['train_list'])                                              # [3차] 샘플링 목록
    counted = files if p['train_list'] else None   # [3차] 가중치는 학습에 쓰는 목록(샘플링 후)에서 센다. None이면 원래 train
    use_fastrcnn_class_weights(DS, p['cls_pw'], counted)                                    # [3차] 손실 가중치. cls_pw가 0이면 원래 손실
    check_applied(train_set, p)
    model = fit(train_set, 1 if SMOKE else p['epochs'], run_dir, tags=['exp3', VARIANT, condition] + (['smoke'] if SMOKE else []))
    use_fastrcnn_class_weights(DS, 0)                                                       # [3차] 다음 조건에 가중치가 남지 않게 되돌린다
    predictions = predict(model, val_set)
    Path(run_dir, 'val_predictions.json').write_text(json.dumps(predictions), encoding='utf-8')   # eval/val_gt.json과 함께 confusion matrix를 그릴 때 쓴다
    save_scores(run_dir, evaluate_val(predictions, f'{DS}/eval/val_gt.json'),
                {'condition': condition, **p, 'sampling': SAMPLING, 'base_epochs': BASE_EPOCHS, 'smoke': SMOKE})

In [ ]:
# Kaggle 제출 (val로 고른 run만)
SUBMIT_RUN = '<run 폴더 이름>'   # SAVE_DIR 아래 run 폴더. 예) v3_frcnn_rfs

from PIL import Image
from torch.utils.data import Dataset
from pill_detection.evaluate import competition_from_classes, write_submission


class TestImages(Dataset):
    """test 이미지 폴더를 predict()가 읽는 형식으로 읽는다. test는 정답이 없어서 COCO 파일 대신 폴더의 PNG를 읽는다."""

    def __init__(self, folder):
        self.paths = sorted(Path(folder).glob('*.png'))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        return to_tensor(Image.open(self.paths[index]).convert('RGB')), {}

    def file_name(self, index):
        return self.paths[index].name   # 1.png처럼 숫자 이름. 숫자가 제출 CSV의 image_id가 된다


!unzip -q -o "{DRIVE_BASE}/data/test_images.zip" -d {REPO}
model = build_model()
model.load_state_dict(torch.load(SAVE_DIR / SUBMIT_RUN / 'weights' / 'last.pt', map_location=DEVICE)['model'])   # 이 노트북은 마지막 epoch의 last.pt만 저장한다
predictions = [p for p in predict(model, TestImages(f'{REPO}/data/raw/test_images')) if p['score'] >= 0.01]   # Ultralytics 제출의 conf=0.01과 같다
n = write_submission(predictions, SAVE_DIR / 'submissions' / f'exp3_{SUBMIT_RUN}.csv',
                     competition_from_classes(f'{DS}/classes.csv'))
print(n, '줄')